In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

from catboost import CatBoostRegressor


In [2]:
# Paths
train_path = "train.csv"
test_path  = "test.csv"

# Read CSV
train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()


Train shape: (15000, 10)
Test shape: (10000, 9)


,id,Sex,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age
0,0,F,1.6000,1.2500,0.5750,43.700754,15.025235,10.262519,13.324265,17.0
1,1,M,1.5750,1.1625,0.3750,32.417653,12.927372,6.208540,8.930093,20.0
2,2,F,1.6125,1.2750,0.4375,34.373769,14.784264,9.142714,10.772810,10.0
3,3,M,1.2250,0.9875,0.3250,18.455524,8.037083,3.926406,4.819415,9.0
4,4,I,1.3750,1.0250,0.3875,22.736299,9.879801,5.513978,6.945627,10.0


In [3]:
# target
y = train["Age"].astype(float)
X = train.drop(columns=["Age"])

# categorical columns
cat_cols = ["Sex"]
cat_features = [X.columns.get_loc(c) for c in cat_cols]  # CatBoost needs column indices

print("Feature columns:", list(X.columns))
print("Categorical columns:", cat_cols)


Feature columns: ['id', 'Sex', 'Length', 'Diameter', 'Height', 'Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight']
Categorical columns: ['Sex']


In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(train), dtype=float)

params = dict(
    loss_function="MAE",
    eval_metric="MAE",
    iterations=8000,
    learning_rate=0.03,
    depth=10,
    l2_leaf_reg=4,
    random_seed=42,
    subsample=0.85,
    rsm=0.85,
    min_data_in_leaf=20,
    od_type="Iter",
    od_wait=300,
    verbose=200
)

params


{'loss_function': 'MAE',
 'eval_metric': 'MAE',
 'iterations': 8000,
 'learning_rate': 0.03,
 'depth': 10,
 'l2_leaf_reg': 4,
 'random_seed': 42,
 'subsample': 0.85,
 'rsm': 0.85,
 'min_data_in_leaf': 20,
 'od_type': 'Iter',
 'od_wait': 300,
 'verbose': 200}

In [ ]:
fold_mae = []
models = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostRegressor(**params)
    model.fit(
        X_tr, y_tr,
        cat_features=cat_features,
        eval_set=(X_va, y_va),
        use_best_model=True
    )

    pred_va = model.predict(X_va)
    oof[va_idx] = pred_va

    mae = mean_absolute_error(y_va, pred_va)
    fold_mae.append(mae)
    models.append(model)

    print(f"Fold {fold} MAE: {mae:.6f}")

cv_mae = mean_absolute_error(y, oof)
print("\nFold MAE list:", [round(m, 6) for m in fold_mae])
print("OOF/CV MAE:", round(cv_mae, 6))


0:	learn: 2.2919114	test: 2.3347722	best: 2.3347722 (0)	total: 392ms	remaining: 52m 12s
